<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Supplementary code for the <a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> book by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Code repository: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>


# 第 2 章练习解答

本 notebook 中使用的包：

In [1]:
from importlib.metadata import version

print("torch version:", version("torch"))
print("tiktoken version:", version("tiktoken"))

torch version: 2.6.0
tiktoken version: 0.9.0


&nbsp;
## 练习 2.1

**练习目的**：理解 BPE 分词器如何处理未见过的单词。BPE 将文本拆分为子词单元（subword），即使单词不在词汇表中，也能正确编码。

**任务**：用 `"Akwirw ier"` 这个伪造的字符串，逐个 token 分析 BPE 的编码/解码过程。

In [2]:
import tiktoken

# 初始化 GPT-2 BPE 分词器
tokenizer = tiktoken.get_encoding("gpt2")

In [3]:
# 将伪造字符串 "Akwirw ier" 编码为 token ID
# 结果 6 个 token：BPE 将未见过的单词拆分为已知的子词
integers = tokenizer.encode("Akwirw ier")
print(integers)

[33901, 86, 343, 86, 220, 959]


In [4]:
# 逐个 token 解码，查看每个 ID 对应的文本片段
# 直观展示 BPE 的子词拆分: Ak|w|ir|w| |ier
for i in integers:
    print(f"{i} -> {tokenizer.decode([i])}")

33901 -> Ak
86 -> w
343 -> ir
86 -> w
220 ->  
959 -> ier


In [5]:
# 逐个验证：每个子词单独编码是否得到相同的 token ID
# "Ak" → [33901]
tokenizer.encode("Ak")

[33901]

In [6]:
# "w" → [86]
tokenizer.encode("w")

[86]

In [7]:
# "ir" → [343]
tokenizer.encode("ir")

[343]

In [8]:
# "w" → [86]
tokenizer.encode("w")

[86]

In [9]:
# " " (空格) → [220]
tokenizer.encode(" ")

[220]

In [10]:
# "ier" → [959]
tokenizer.encode("ier")

[959]

In [11]:
# 验证 roundtrip：6 个 token ID 解码后还原为原始字符串 "Akwirw ier"
# 结论：即使单词从未见过，BPE 也能无损地编码/解码
tokenizer.decode([33901, 86, 343, 86, 220, 959])

'Akwirw ier'

&nbsp;
## 练习 2.2

**练习目的**：理解滑动窗口参数（`max_length` 和 `stride`）对数据集形状的影响。

**任务**：
- 用 `batch_size=4, max_length=2, stride=2` 创建 DataLoader，观察输出
- 改为 `batch_size=4, max_length=8, stride=2`，对比输出差异

In [12]:
# 练习 2.2 用的精简版 DataLoader（去掉了 shuffle/drop_last 参数）
import tiktoken
import torch
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]


def create_dataloader(txt, batch_size=4, max_length=256, stride=128):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    dataloader = DataLoader(dataset, batch_size=batch_size)
    return dataloader


# 读取文本并编码
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

tokenizer = tiktoken.get_encoding("gpt2")
encoded_text = tokenizer.encode(raw_text)

In [13]:
# 实验 A: max_length=2, stride=2（每段 2 个 token，无重叠）
# 输出 x shape (4, 2) = 4 个样本 × 2 个 token
#   [[  40,  367],     ← token[0:2]
#    [2885, 1464],     ← token[2:4]
#    [1807, 3619],     ← token[4:6]
#    [ 402,  271]]     ← token[6:8]
dataloader = create_dataloader(raw_text, batch_size=4, max_length=2, stride=2)

for batch in dataloader:
    x, y = batch
    break

x

tensor([[  40,  367],
        [2885, 1464],
        [1807, 3619],
        [ 402,  271]])

In [14]:
# 实验 B: max_length=8, stride=2（每段 8 个 token，窗口重叠 6 个）
# 输出 x shape (4, 8) = 4 个样本 × 8 个 token
#   每行仍然有 8 个 token，但行与行之间只偏移 2 步（stride=2）
#   重叠 = max_length - stride = 8 - 2 = 6 个 token
# 对比实验 A: 窗口更长(8 vs 2) + 有重叠(stride=2 < max_length=8)
dataloader = create_dataloader(raw_text, batch_size=4, max_length=8, stride=2)

for batch in dataloader:
    x, y = batch
    break

x

tensor([[   40,   367,  2885,  1464,  1807,  3619,   402,   271],
        [ 2885,  1464,  1807,  3619,   402,   271, 10899,  2138],
        [ 1807,  3619,   402,   271, 10899,  2138,   257,  7026],
        [  402,   271, 10899,  2138,   257,  7026, 15632,   438]])